# 🩺 Multilingual Health QA — Low-Resource African Languages
### Zindi Competition Baseline Notebook

**Approach:** Fine-tune `google/mt5-small` (multilingual T5) on the training Q&A pairs, then generate answers for the test set.  
**Metrics:** ROUGE-1 F1, ROUGE-L F1, LLM-as-a-Judge (normalized 0–1)  
**Languages:** Luganda, Kiswahili, Akan, Amharic (and others in dataset)

---
> ⚡ **Runtime:** Set runtime to **GPU (T4)** before running — Runtime → Change runtime type → T4 GPU

## 1. Install Dependencies

In [ ]:
%%capture
!pip install transformers datasets rouge-score sentencepiece accelerate pandas numpy scikit-learn
!pip install -q bitsandbytes  # for optional 8-bit quantization

## 2. Mount Google Drive & Upload Data

Download the competition data from Zindi, then upload it to your Google Drive under `MyDrive/zindi_health_qa/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/zindi_health_qa/'
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Data directory: {DATA_DIR}")
print("Files found:", os.listdir(DATA_DIR))

## 3. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import torch
import os
import json
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict
from rouge_score import rouge_scorer

# ─── CONFIGURATION ───────────────────────────────────────────────────────────
# Experiment: Baseline — mt5-small, no PEFT, full fine-tune
EXPERIMENT_NAME = "exp01_mt5_small_baseline"
MODEL_NAME      = "google/mt5-small"   # swap to mt5-base for Exp 2
MAX_INPUT_LEN   = 128
MAX_TARGET_LEN  = 256
BATCH_SIZE      = 8
GRAD_ACCUM      = 4    # effective batch = 32
LEARNING_RATE   = 5e-4
NUM_EPOCHS      = 5
WARMUP_RATIO    = 0.1
SEED            = 42

OUTPUT_DIR = f"/content/drive/MyDrive/zindi_health_qa/outputs/{EXPERIMENT_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 4. Load & Explore the Dataset

In [ ]:
# ─── Load files (adjust filenames to match what Zindi provides) ───────────────
train_df = pd.read_csv(os.path.join(DATA_DIR, 'Train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'Test.csv'))
ss_df    = pd.read_csv(os.path.join(DATA_DIR, 'SampleSubmission.csv'))

print(f"Train shape : {train_df.shape}")
print(f"Test shape  : {test_df.shape}")
print(f"\nTrain columns: {train_df.columns.tolist()}")
print(f"Test  columns: {test_df.columns.tolist()}")
train_df.head(5)

In [ ]:
# ─── Identify column names (adjust if different in actual data) ───────────────
# Expected columns based on competition description:
#   ID, Language, Question, Answer  (train)
#   ID, Language, Question          (test)

ID_COL       = 'ID'
LANG_COL     = 'Language'   # may be 'language' — check and adjust
QUESTION_COL = 'Question'   # may be 'question'
ANSWER_COL   = 'Answer'     # may be 'answer'

# Normalize column names to lowercase for safety
train_df.columns = [c.strip() for c in train_df.columns]
test_df.columns  = [c.strip() for c in test_df.columns]

# Language distribution
print("\n📊 Language distribution in train:")
print(train_df[LANG_COL].value_counts())

# Answer length stats
train_df['answer_len'] = train_df[ANSWER_COL].str.split().str.len()
print(f"\n📏 Answer length stats (words):")
print(train_df['answer_len'].describe())

In [ ]:
# ─── Quick sanity check: show one example per language ───────────────────────
for lang in train_df[LANG_COL].unique():
    sample = train_df[train_df[LANG_COL] == lang].iloc[0]
    print(f"\n[{lang}]")
    print(f"  Q: {sample[QUESTION_COL][:120]}")
    print(f"  A: {sample[ANSWER_COL][:200]}")

## 5. Preprocessing

In [ ]:
import re

def clean_text(text):
    """Basic text cleaning — preserve multilingual characters."""
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)       # collapse multiple spaces
    text = re.sub(r'[\x00-\x08\x0b-\x1f]', '', text)  # remove control chars
    return text

def build_prompt(row, include_language=True):
    """
    Prompt format fed to the encoder.
    Experiment variable: try with/without language tag.
    """
    q = clean_text(row[QUESTION_COL])
    if include_language:
        lang = clean_text(row[LANG_COL])
        return f"answer health question in {lang}: {q}"
    return f"answer health question: {q}"

# Apply to train and test
train_df['input_text']  = train_df.apply(build_prompt, axis=1)
train_df['target_text'] = train_df[ANSWER_COL].apply(clean_text)
test_df['input_text']   = test_df.apply(build_prompt, axis=1)

print("Sample input prompt:")
print(train_df['input_text'].iloc[0])
print("\nTarget answer:")
print(train_df['target_text'].iloc[0])

## 6. Train / Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

# Stratify by language to keep balanced validation set
train_data, val_data = train_test_split(
    train_df,
    test_size=0.1,
    random_state=SEED,
    stratify=train_df[LANG_COL]
)

print(f"Train: {len(train_data)} | Val: {len(val_data)}")
print("\nVal language distribution:")
print(val_data[LANG_COL].value_counts())

## 7. Tokenize

In [ ]:
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    model_inputs = tokenizer(
        examples['input_text'],
        max_length=MAX_INPUT_LEN,
        padding=False,
        truncation=True
    )
    labels = tokenizer(
        text_target=examples['target_text'],
        max_length=MAX_TARGET_LEN,
        padding=False,
        truncation=True
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

def df_to_hf_dataset(df, include_target=True):
    cols = ['input_text']
    if include_target:
        cols.append('target_text')
    return Dataset.from_pandas(df[cols].reset_index(drop=True))

train_hf = df_to_hf_dataset(train_data).map(tokenize_fn, batched=True, remove_columns=['input_text', 'target_text'])
val_hf   = df_to_hf_dataset(val_data).map(tokenize_fn, batched=True, remove_columns=['input_text', 'target_text'])

print(f"Tokenized train: {len(train_hf)} examples")
print(f"Tokenized val  : {len(val_hf)} examples")
print(f"Features: {train_hf.features}")

## 8. Load Model

In [ ]:
print(f"Loading model: {MODEL_NAME}")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable:,}")

## 9. Evaluation Metric (ROUGE)

In [ ]:
from rouge_score import rouge_scorer as rs_module

scorer = rs_module.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Replace -100 (padding) in labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds,   skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels,  skip_special_tokens=True)

    r1_scores, rL_scores = [], []
    for pred, label in zip(decoded_preds, decoded_labels):
        pred  = pred.strip()
        label = label.strip()
        scores = scorer.score(label, pred)
        r1_scores.append(scores['rouge1'].fmeasure)
        rL_scores.append(scores['rougeL'].fmeasure)

    return {
        'rouge1': round(np.mean(r1_scores), 4),
        'rougeL': round(np.mean(rL_scores), 4),
    }

## 10. Training

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rouge1",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
    seed=SEED,
    save_total_limit=2,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print(f"🚀 Starting training — Experiment: {EXPERIMENT_NAME}")
trainer.train()

## 11. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history

train_loss = [(l['epoch'], l['loss']) for l in logs if 'loss' in l and 'eval_loss' not in l]
eval_rouge1 = [(l['epoch'], l['eval_rouge1']) for l in logs if 'eval_rouge1' in l]
eval_rougeL = [(l['epoch'], l['eval_rougeL']) for l in logs if 'eval_rougeL' in l]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
if train_loss:
    epochs_l, losses = zip(*train_loss)
    axes[0].plot(epochs_l, losses, 'b-o', markersize=3)
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)

# Validation ROUGE
if eval_rouge1:
    e1, r1 = zip(*eval_rouge1)
    eL, rL = zip(*eval_rougeL)
    axes[1].plot(e1, r1, 'g-o', label='ROUGE-1', markersize=5)
    axes[1].plot(eL, rL, 'r-s', label='ROUGE-L', markersize=5)
    axes[1].set_title('Validation ROUGE Scores')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('F1 Score')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Experiment: {EXPERIMENT_NAME}', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved training curves")

## 12. Per-Language Evaluation on Validation Set

In [ ]:
def evaluate_per_language(model, tokenizer, df, device):
    """Run inference and compute ROUGE per language."""
    model.eval()
    results = []

    for _, row in df.iterrows():
        inp = tokenizer(
            row['input_text'],
            return_tensors='pt',
            max_length=MAX_INPUT_LEN,
            truncation=True
        ).to(device)

        with torch.no_grad():
            out = model.generate(
                **inp,
                max_new_tokens=MAX_TARGET_LEN,
                num_beams=4,
                early_stopping=True
            )
        pred = tokenizer.decode(out[0], skip_special_tokens=True).strip()
        ref  = row['target_text'].strip()

        s = scorer.score(ref, pred)
        results.append({
            'language': row[LANG_COL],
            'rouge1':   s['rouge1'].fmeasure,
            'rougeL':   s['rougeL'].fmeasure,
            'pred':     pred,
            'ref':      ref
        })

    return pd.DataFrame(results)

print("Running per-language eval on validation set...")
# Use a sample for speed during baseline exploration
val_sample = val_data.groupby(LANG_COL).apply(lambda x: x.sample(min(20, len(x)), random_state=SEED)).reset_index(drop=True)
eval_df = evaluate_per_language(model, tokenizer, val_sample, device)

print("\n📊 ROUGE scores per language:")
per_lang = eval_df.groupby('language')[['rouge1', 'rougeL']].mean().round(4)
print(per_lang)
print(f"\nOverall ROUGE-1: {eval_df['rouge1'].mean():.4f}")
print(f"Overall ROUGE-L: {eval_df['rougeL'].mean():.4f}")

In [ ]:
# Visualize per-language ROUGE
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(per_lang))
width = 0.35
ax.bar([i - width/2 for i in x], per_lang['rouge1'], width, label='ROUGE-1', color='steelblue')
ax.bar([i + width/2 for i in x], per_lang['rougeL'], width, label='ROUGE-L', color='coral')
ax.set_xticks(list(x))
ax.set_xticklabels(per_lang.index, rotation=15)
ax.set_ylabel('F1 Score')
ax.set_title(f'Per-Language ROUGE — {EXPERIMENT_NAME}')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'per_language_rouge.png'), dpi=150)
plt.show()

## 13. Generate Test Predictions

In [ ]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

def generate_predictions(model, tokenizer, texts, device, batch_size=16):
    """Batched inference with beam search."""
    model.eval()
    all_preds = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Generating"):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch,
            return_tensors='pt',
            max_length=MAX_INPUT_LEN,
            truncation=True,
            padding=True
        ).to(device)

        with torch.no_grad():
            out = model.generate(
                **enc,
                max_new_tokens=MAX_TARGET_LEN,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=3
            )
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        all_preds.extend([d.strip() for d in decoded])

    return all_preds

print("Generating test predictions...")
test_preds = generate_predictions(model, tokenizer, test_df['input_text'].tolist(), device)
print(f"Generated {len(test_preds)} predictions")

# Quick sanity check
for i in range(3):
    print(f"\n[{i}] Q: {test_df['input_text'].iloc[i][:100]}")
    print(f"     A: {test_preds[i][:200]}")

## 14. Build Submission File

In [ ]:
# The submission requires 4 columns: ID, TargetRLF1, TargetR1F1, TargetLLM
# All three Target columns should contain the SAME answer string

submission = pd.DataFrame({
    'ID':         test_df[ID_COL].values,
    'TargetRLF1': test_preds,
    'TargetR1F1': test_preds,
    'TargetLLM':  test_preds
})

# Validate against sample submission
print(f"Submission shape: {submission.shape}")
print(f"Sample submission shape: {ss_df.shape}")
assert set(submission['ID']) == set(ss_df['ID']), "❌ ID mismatch with sample submission!"
print("✅ IDs match sample submission")

sub_path = os.path.join(OUTPUT_DIR, f'submission_{EXPERIMENT_NAME}.csv')
submission.to_csv(sub_path, index=False)
print(f"\n✅ Submission saved to: {sub_path}")
submission.head()

## 15. Save Experiment Log

In [ ]:
# Log experiment metadata for your report
import json

best_eval = max(
    [l for l in trainer.state.log_history if 'eval_rouge1' in l],
    key=lambda x: x['eval_rouge1'],
    default={}
)

experiment_log = {
    "experiment":        EXPERIMENT_NAME,
    "model":             MODEL_NAME,
    "max_input_len":     MAX_INPUT_LEN,
    "max_target_len":    MAX_TARGET_LEN,
    "batch_size":        BATCH_SIZE,
    "grad_accum":        GRAD_ACCUM,
    "learning_rate":     LEARNING_RATE,
    "num_epochs":        NUM_EPOCHS,
    "warmup_ratio":      WARMUP_RATIO,
    "prompt_strategy":   "with_language_tag",
    "beam_search_beams": 4,
    "val_rouge1":        best_eval.get('eval_rouge1', 'N/A'),
    "val_rougeL":        best_eval.get('eval_rougeL', 'N/A'),
    "leaderboard_score": "TBD — submit and update",
    "notes":             "Baseline: full fine-tune, no PEFT, mt5-small"
}

log_path = os.path.join(OUTPUT_DIR, 'experiment_log.json')
with open(log_path, 'w') as f:
    json.dump(experiment_log, f, indent=2)

print("📝 Experiment log:")
print(json.dumps(experiment_log, indent=2))

## 16. Next Experiment Ideas

Use this checklist to plan your 10+ experiments for the report:

| # | Experiment | What Changes | Expected Effect |
|---|---|---|---|
| 1 | **Baseline** *(this notebook)* | mt5-small, full fine-tune | Reference point |
| 2 | Larger model | `google/mt5-base` | Better capacity |
| 3 | No language tag in prompt | Remove `in {lang}:` prefix | Test prompt impact |
| 4 | LoRA / PEFT | Add LoRA adapters | Fewer trainable params, faster |
| 5 | Longer answers | Increase `MAX_TARGET_LEN` to 512 | Catch truncated targets |
| 6 | More beams | `num_beams=8` at inference | Better decoding |
| 7 | Length penalty | `length_penalty=1.5` | Longer, more complete answers |
| 8 | `facebook/nllb-200-distilled-600M` | Different base model | NLLB covers many African languages |
| 9 | Data augmentation | Back-translate underrepresented languages | More data for low-resource langs |
| 10 | Ensemble / voting | Average outputs of 2 models | Combine strengths |

---
> 💡 After each run: record val ROUGE + leaderboard score in `experiment_log.json`  
> 💡 Commit code with a meaningful message after each experiment for clean GitHub history